# 5.2. 参数管理

本节介绍如何访问参数、初始化参数，以及在模型组件间共享参数。请自己完成每个代码单元。

导入 torch、torch.nn；定义单隐藏层多层感知机 net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))；生成 X = torch.rand(size=(2, 4)) 并调用 net(X)。

In [1]:
import torch
from torch import nn
net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X=torch.rand(2,4)
net(X)

tensor([[0.1700],
        [0.1172]], grad_fn=<AddmmBackward0>)

### 5.2.1. 参数访问

用索引 net[i] 访问第 i 层（0 从输入开始）。尝试：
- print(net[2].state_dict()) 用字典查看第 2 个全连接层参数
- 访问目标参数：type(net[2].bias) / net[2].bias / net[2].bias.data，以及 net[2].weight.grad == None
- 一次性访问所有参数：for name, param in net.named_parameters(): print(name, param.shape)；以及 net.state_dict()['2.bias'].data

In [2]:
print(net[2].state_dict())
net[2].weight.grad==False

OrderedDict({'weight': tensor([[-0.2689,  0.1756,  0.2394,  0.1560,  0.2557,  0.2851, -0.2059, -0.0326]]), 'bias': tensor([0.0896])})


False

### 5.2.1.3. 从嵌套块收集参数

定义两个"块工厂"函数并嵌套组合：
- def block1(): return nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4), nn.ReLU())
- def block2(): 建立 net = nn.Sequential()，for i in range(4): net.add_module(f'block {i}', block1())，return net
- rgnet = nn.Sequential(block2(), nn.Linear(4, 1))，用 X 调用

再尝试 print(rgnet) 观察结构，以及访问嵌套层参数如 rgnet[0][1][0].bias.data。

In [7]:
def block1():
    return nn.Sequential(nn.Linear(4, 8),nn.ReLU(),nn.Linear(8,4),nn.ReLU())
def block2():
    net=nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return  net

rgnet=nn.Sequential(block2(),nn.Linear(4,1))
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


### 5.2.2. 参数初始化

#### 内置初始化

写一个初始化函数并用 net.apply 应用（每个模块会被调用）：
- def init_normal(m): if type(m) == nn.Linear: nn.init.normal_(m.weight, mean=0, std=0.01); nn.init.zeros_(m.bias)
- 用 net.apply(init_normal) 后查看 net[0].weight.data

也可换其他初始化方式（如 nn.init.constant_、nn.init.xavier_uniform_）再比较。

In [9]:
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data


tensor([[ 0.0195,  0.0135,  0.0048,  0.0055],
        [ 0.0066,  0.0005,  0.0028,  0.0067],
        [ 0.0043, -0.0070, -0.0023,  0.0111],
        [ 0.0156, -0.0026,  0.0038, -0.0023],
        [ 0.0022,  0.0019,  0.0009, -0.0050],
        [-0.0025,  0.0155, -0.0121, -0.0079],
        [-0.0096,  0.0067, -0.0092, -0.0130],
        [ 0.0010, -0.0256,  0.0103,  0.0036]])

#### 自定义初始化

写一个自定义初始化函数 my_init：
- 用 python 计算（如对标准均匀分布采样，再按条件调整数值），例如 def my_init(m): if type(m) == nn.Linear: nn.init.uniform_(m.weight, -10, 10); m.weight.data *= m.weight.data.abs() >= 5（只保留绝对值>=5的元素）
- 用 net.apply(my_init) 并查看结果

In [14]:
def my_init(m):
    if type(m)==nn.Linear:
        nn.init.normal(m.weight, -10, 10)
        m.weight.data *=m.weight.data.abs() >= 5
net.apply(my_init)
print(net[0].weight.data)

tensor([[-21.3912, -16.2033, -17.6652, -10.2464],
        [-24.7532, -11.6992, -30.4343, -14.6877],
        [ -9.9950,   8.7330,  -0.0000,  -7.1561],
        [ -0.0000,  -9.2486,  -0.0000,  -8.5661],
        [-14.9765,   0.0000,  -6.9145,   0.0000],
        [ 18.3615,  -7.3381,  -0.0000,  -7.7988],
        [ -8.3177,  -0.0000,   0.0000,  -8.7119],
        [-24.5251,   0.0000,  -6.1971, -19.9591]])


C:\Users\16608\AppData\Local\Temp\ipykernel_26116\1233966530.py:3: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  nn.init.normal(m.weight, -10, 10)


#### 参数绑定

在同一模型的不同位置共享同一参数：
- shared = nn.Linear(8, 8)
- net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), shared, nn.ReLU(), shared, nn.ReLU(), nn.Linear(8, 1))
- 用 X 调用后检查各层参数：print(net[2].weight.data == net[4].weight.data)。再思考：参数绑定（共享）的机制与注意点。

In [15]:
shared=nn.Linear(8,8)
net=nn.Sequential(nn.Linear(4,8),nn.ReLU(),shared,nn.ReLU(),shared, nn.ReLU(), nn.Linear(8, 1))
print(net[2].weight.data == net[4].weight.data)

tensor([[True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True]])


### 5.2.4. 小结

- 可以通过索引或名字访问模型参数，也可一次性枚举所有命名参数
- 参数可以通过内置或自定义初始化函数进行初始化
- 不同模型组件可以共享参数

### 5.2.5. 练习

1. 尝试用深度不同、宽度不同的多层感知机检查参数数量与形状。
2. 自定义一个初始化函数，可能是更复杂的分布或条件处理。
3. 尝试共享一个层的参数到两个不同的位置，观察梯度更新。